# Description: Gather results and create final data structures

This notebook consolidates scan-level outputs produced by earlier processing notebooks into dataset-level summary artifacts under `code/notebooks/summary_files/`.
The goal is to create canonical inputs for downstream QC, figure-generation, and CPM notebooks so they do not need to repeatedly recompute scan-level metrics.

If any upstream notebook regenerates FC, pBOLD, TSNR, or Tedana outputs, rerun this notebook so downstream analyses read refreshed summary files.

## Artifacts handled in this notebook

1. `{DATASET}_FC.pkl` (dictionary of FC/cov matrices per scan/scenario)
2. `{DATASET}_pBOLD.nc` (xarray DataArray of pBOLD values)
3. `{DATASET}_TSNR.pkl` (dictionary of TSNR DataFrames)
4. `{DATASET}_Tedana_QC.pkl` (dictionary of ICA component QC DataFrames)
5. `physio_dict` intended for `{DATASET}_Physiological_Timeseries.pkl` (physiology time series dictionary)

discovery_FC.pkl  discovery_GS_info_and_ts.pkl  discovery_ICAs.pkl  discovery_pBOLD.nc  discovery_Physiological_Regressors.pkl  discovery_Physiological_Timeseries.pkl  discovery_Tedana_QC.pkl  discovery_TSNR.pkl


In [1]:
import os.path as osp
import numpy as np
import pandas as pd
import xarray as xr
import pickle
from utils.basics import ATLASES_DIR, CODE_DIR, PRCS_DATA_DIR, DOWNLOAD_DIRS, FMRI_TRS, FMRI_FINAL_NUM_SAMPLES, NUM_DISCARDED_VOLUMES
from utils.basics import get_dataset_index, echo_pairs_tuples, get_altas_info, pairs_of_echo_pairs
from tqdm.notebook import tqdm

from afnipy import lib_physio_reading as lpr
from afnipy import lib_physio_opts    as lpo
from afnipy.lib_afni1D import Afni1D
import copy

## Configuration: Select dataset

Choose `discovery` or `evaluation`. The selected dataset controls both the scan index and the filesystem roots used throughout the notebook.


In [2]:
DATASET = input ('Select Dataset (discovery or evaluation):')
DOWNLOAD_DIR = DOWNLOAD_DIRS[DATASET]

## Configuration: Define preprocessing scenarios

`pp_opts` maps human-readable labels to canonical pipeline IDs used in filenames and dictionary/xarray keys.
All downstream summary objects index data by these canonical IDs (e.g., `ALL_Basic`, `ALL_GS`, `ALL_tedana-fastica`).


In [3]:
pp_opts = {'No Censoring | No Regression':'ALL_NoRegression',
           'No Censoring | Basic':'ALL_Basic',
           'No Censoring | GSR':'ALL_GS',
           'No Censoring | Tedana (fastica)':'ALL_Tedana-fastica'}

## Configuration: Atlas metadata

Build the dataset-specific atlas path (`Power264-{DATASET}`), then load ROI metadata.
`roi_idxs` (MultiIndex over ROI name/id/hemisphere/network) is reused as the row/column index in FC matrices.


In [4]:
ATLAS_NAME = f'Power264-{DATASET}'
ATLAS_DIR = osp.join(ATLASES_DIR,ATLAS_NAME)

In [5]:
roi_info_df, _ = get_altas_info(ATLAS_DIR,ATLAS_NAME)
roi_idxs = roi_info_df.set_index(['ROI_Name', 'ROI_ID', 'Hemisphere', 'Network']).index

++ INFO [get_nw_cmap]: Gathering ROI information from file /data/SFIMJGC_HCP7T/BCBL2024/atlases/Power264-evaluation/Power264-evaluation.roi_info.csv
++ INFO: Number of ROIs = 203 | Number of Connections = 20503


## Configuration: Enumerate scans

`ds_index` provides the `(Subject, Session)` combinations processed by all sections below.
`ses_list` and `sbj_list` are later used as coordinate axes when initializing xarray objects.


In [6]:
ds_index = get_dataset_index(DATASET)
ses_list = list(ds_index.get_level_values('Session').unique())
sbj_list = list(ds_index.get_level_values('Subject').unique())

++ Number of scans    = 439
++ Number of subjects = 221


---
# 1. Gather Functional Connectivity Estimates

This section loads ROI time series for each scan/scenario/echo-pair and computes:
- Pearson correlation matrices (`'corr'`)
- Covariance matrices (`'cov'`)

Both are stored in a dictionary keyed by scan + processing context.


In [7]:
fc_path = osp.join(CODE_DIR,'notebooks','summary_files',f'{DATASET}_FC.pkl')
print('++ FC will be saved in %s' % fc_path)

++ FC will be saved in /data/SFIMJGC_HCP7T/BCBL2024/me_staticfc/code/notebooks/summary_files/evaluation_FC.pkl


In [8]:
%%time
data_fc = {}
print(f"++ INFO: Loading all FC matrices for the {DATASET} into memory...")
for sbj,ses in tqdm(list(ds_index)):
    for nordic in ['off','on']:
        for pp in pp_opts.values():
            for (e_x,e_y) in echo_pairs_tuples:
                d_folder = f'D03_Preproc_{ses}_NORDIC-{nordic}'
                # Compose path to input TS
                roi_ts_path_x = osp.join(PRCS_DATA_DIR,sbj,d_folder,f'errts.{sbj}.r01.{e_x}.volreg.spc.tproject_{pp}.{ATLAS_NAME}_000.netts')
                roi_ts_path_y = osp.join(PRCS_DATA_DIR,sbj,d_folder,f'errts.{sbj}.r01.{e_y}.volreg.spc.tproject_{pp}.{ATLAS_NAME}_000.netts')
                # Load TS into memory
                if (not osp.exists(roi_ts_path_x)) or (not osp.exists(roi_ts_path_y)):
                    print(f'++ WARNING: Missing input files for {sbj},{ses},{e_x},{e_y},{nordic},{pp}')
                    print(f'            {roi_ts_path_x}')
                    print(f'            {roi_ts_path_y}')
                    i+=1
                    continue
                roi_ts_x      = np.loadtxt(roi_ts_path_x)
                roi_ts_y      = np.loadtxt(roi_ts_path_y)
                aux_ts_x = pd.DataFrame(roi_ts_x, columns=roi_info_df['ROI_Name'].values)
                aux_ts_y = pd.DataFrame(roi_ts_y, columns=roi_info_df['ROI_Name'].values)
                # Compute the full correlation matrix between aux_ts_x and aux_ts_y
                aux_r    = np.corrcoef(aux_ts_x.T, aux_ts_y.T)[:aux_ts_x.shape[1], aux_ts_x.shape[1]:]
                aux_c    = np.cov(aux_ts_x.T, aux_ts_y.T)[:aux_ts_x.shape[1], aux_ts_x.shape[1]:]
                data_fc[sbj, ses, pp,nordic,'|'.join((e_x,e_y)),'corr']  = pd.DataFrame(aux_r,index=roi_idxs,columns=roi_idxs)
                data_fc[sbj, ses, pp,nordic,'|'.join((e_x,e_y)),'cov']  = pd.DataFrame(aux_c,index=roi_idxs,columns=roi_idxs)
print('++ INFO: Saving to disk...',end='')
with open(fc_path, 'wb') as f:
    pickle.dump(data_fc, f)
print('[DONE]')

++ INFO: Loading all FC matrices for the evaluation into memory...


  0%|          | 0/439 [00:00<?, ?it/s]

++ INFO: Saving to disk...[DONE]
CPU times: user 4min 31s, sys: 1min 3s, total: 5min 34s
Wall time: 6min 58s


### Output Schema: `{DATASET}_FC.pkl`

- **Path**: `code/notebooks/summary_files/{DATASET}_FC.pkl`
- **Serialized object type**: `dict`
- **Dictionary key (6-tuple)**:
  - `(Subject, Session, Pre-processing, NORDIC, ee_vs_ee, fc_metric)`
  - Example: `('sub-001', 'ses-1', 'ALL_Basic', 'off', 'e01|e02', 'R')`
- **Key domains**:
  - `Subject`: from `ds_index`
  - `Session`: from `ds_index`
  - `Pre-processing`: values of `pp_opts`
  - `NORDIC`: `{'off', 'on'}`
  - `ee_vs_ee`: entries from `echo_pairs_tuples` joined with `'|'`
  - `fc_metric`: `{'corr', 'cov'}` for correlation/covariance
- **Dictionary value**: `pandas.DataFrame` of shape `(N_ROIs, N_ROIs)`
  - `index` and `columns`: `roi_idxs` MultiIndex with levels `['ROI_Name', 'ROI_ID', 'Hemisphere', 'Network']`

### Downstream notebooks that load this file

- `N09_Figure03_pBOLDonDiscovery.ipynb`
- `N11_Figure05_ScanLevel_ProblemIdentification.ipynb`
- `N13_RunCPMAnalyses.ipynb`


---
# 2. Gather pBOLD estimates

This section assembles all pBOLD CSV outputs into a single 7D xarray object (`pBOLD_xr`) and persists it as NetCDF.


In [9]:
pBOLD_xr_path = osp.join(CODE_DIR,'notebooks','summary_files',f'{DATASET}_pBOLD.nc')
print('++ pBOLD will be saved in %s' % pBOLD_xr_path)

++ pBOLD will be saved in /data/SFIMJGC_HCP7T/BCBL2024/me_staticfc/code/notebooks/summary_files/evaluation_pBOLD.nc


In [10]:
pBOLD_xr = xr.DataArray(dims=['sbj','ses','pp','nordic','fc_metric','ee_vs_ee','qc_metric',],
                         coords={'sbj':       sbj_list,
                                 'ses':       ses_list,
                                 'pp':        list(pp_opts.values()),
                                 'nordic':    ['off','on'],
                                 'fc_metric': ['corr','cov'],
                                 'ee_vs_ee':  pairs_of_echo_pairs+['scan'],
                                 'qc_metric': ['pBOLD']})

In [11]:
%%time
print(f"++ INFO: Loading all pBOLD estimates for the {DATASET} into memory...")
for sbj,ses in tqdm(list(ds_index)):
    for NORDIC in ['off','on']:
            for pp in pp_opts.values():
                for fc_metric in ['corr','cov']:
                    pBOLD_path    = osp.join(PRCS_DATA_DIR,sbj,f'D03_Preproc_{ses}_NORDIC-{NORDIC}',f'errts.{sbj}.r01.volreg.spc.tproject_{pp}.{ATLAS_NAME}_000.pBOLD_{fc_metric}.csv')
                    try:
                        pBOLD = pd.read_csv(pBOLD_path)
                    except:
                        print('Problematic file: %s' % pBOLD_path )
                    pBOLD_xr.loc[sbj,ses,pp,NORDIC,fc_metric,pBOLD['ee_vs_ee'].to_list()] = pBOLD['pBOLD'].values.reshape(len(pairs_of_echo_pairs)+1,1)

++ INFO: Loading all pBOLD estimates for the evaluation into memory...


  0%|          | 0/439 [00:00<?, ?it/s]

CPU times: user 13.4 s, sys: 957 ms, total: 14.3 s
Wall time: 26.4 s


In [12]:
%%time
print('++ INFO: Saving to disk...',end='')
pBOLD_xr.to_netcdf(pBOLD_xr_path)
print('[DONE]')

++ INFO: Saving to disk...[DONE]
CPU times: user 27.2 ms, sys: 7.82 ms, total: 35 ms
Wall time: 134 ms


### Output Schema: `{DATASET}_pBOLD.nc`

- **Path**: `code/notebooks/summary_files/{DATASET}_pBOLD.nc`
- **Serialized object type**: `xarray.DataArray`
- **Dimensions**:
  - `['sbj', 'ses', 'pp', 'nordic', 'fc_metric', 'ee_vs_ee', 'qc_metric']`
- **Coordinates**:
  - `sbj`: `sbj_list`
  - `ses`: `ses_list`
  - `pp`: `list(pp_opts.values())`
  - `nordic`: `['off', 'on']`
  - `fc_metric`: `['corr', 'cov']`
  - `ee_vs_ee`: `pairs_of_echo_pairs + ['scan']`
  - `qc_metric`: `['pBOLD']`
- **Stored value semantics**:
  - Scalar pBOLD estimate for each coordinate combination.

### Downstream notebooks that load this file

- `N09_Figure03_pBOLDonDiscovery.ipynb`
- `N10a_TSNR_and_pBOLD_BarPlots_Figure04.evaluation.ipynb`
- `N10b_TSNR_and_pBOLD_BarPlots_SuppFig03.discovery.ipynb`
- `N11_Figure05_ScanLevel_ProblemIdentification.ipynb`


---
# 3. Gather TSNR estimates

This section collects TSNR summary values (full-brain and visual-cortex) into DataFrames, then stores them in a dictionary keyed by metric context.


In [13]:
TSNR_path = osp.join(CODE_DIR,'notebooks','summary_files',f'{DATASET}_TSNR.pkl')
print('++ TSNR will be saved in %s' % TSNR_path)

++ TSNR will be saved in /data/SFIMJGC_HCP7T/BCBL2024/me_staticfc/code/notebooks/summary_files/evaluation_TSNR.pkl


In [14]:
%%time
TSNR = {}
print(f"++ INFO: Loading all FC matrices for the {DATASET} into memory...")
for TSNR_metric in ['TSNR (Full Brain)','TSNR (Visual Cortex)']:
    aux_df = pd.DataFrame(columns=['Subject','Session','Pre-processing','m-NORDIC',TSNR_metric])
    aux_df.set_index(['Subject','Session','Pre-processing','m-NORDIC'], inplace=True)
    for sbj,ses in tqdm(list(ds_index), desc=TSNR_metric):
        partial_key = (sbj, ses)
        sbj_ses_in_fc = any(key[:len(partial_key)] == partial_key for key in data_fc)
        if not sbj_ses_in_fc:
            print('++ WARNING: This combination of sbj,ses [%s,%s] is not available. XR will contain np.nan.' % (sbj,ses))
            continue
        for pp in pp_opts.values():
            for nordic in ['off','on']:            
                d_folder = f'D03_Preproc_{ses}_NORDIC-{nordic}'
                if TSNR_metric == 'TSNR (Visual Cortex)':
                    aux_rois_path = osp.join(PRCS_DATA_DIR,sbj,d_folder,'tsnr_stats_regress',f'TSNR_ROIs_e02_{pp}.txt')
                    aux_rois      = pd.read_csv(aux_rois_path,skiprows=3, sep=r'\s+').drop(0).set_index('ROI_name')
                    aux_df.loc[sbj,ses,pp,nordic] = float(aux_rois.loc['GHCP-R_Primary_Visual_Cortex','Tmed'])
                if TSNR_metric == 'TSNR (Full Brain)':
                    aux_fb_path   = osp.join(PRCS_DATA_DIR,sbj,d_folder,'tsnr_stats_regress',f'TSNR_FB_e02_{pp}.txt')
                    aux_fb        = pd.read_csv(aux_fb_path,skiprows=3, sep=r'\s+').drop(0).set_index('ROI_name')
                    aux_df.loc[sbj,ses,pp,nordic] = float(aux_fb.loc['NONE','Tmed'])
    TSNR['cov',TSNR_metric] = aux_df.reset_index()
    TSNR['corr',TSNR_metric] = aux_df.reset_index()
print('++ INFO: Saving to disk...',end='')
with open(TSNR_path, 'wb') as f:
    pickle.dump(TSNR, f)
print('[DONE]')

++ INFO: Loading all FC matrices for the evaluation into memory...


TSNR (Full Brain):   0%|          | 0/439 [00:00<?, ?it/s]

TSNR (Visual Cortex):   0%|          | 0/439 [00:00<?, ?it/s]

++ INFO: Saving to disk...[DONE]
CPU times: user 35.8 s, sys: 1.35 s, total: 37.1 s
Wall time: 52.8 s


### Output Schema: `{DATASET}_TSNR.pkl`

- **Path**: `code/notebooks/summary_files/{DATASET}_TSNR.pkl`
- **Serialized object type**: `dict`
- **Dictionary key (2-tuple)**:
  - `(fc_metric, tsnr_metric)`
  - `fc_metric` in `{'cov', 'corr'}`
  - `tsnr_metric` in `{'TSNR (Full Brain)', 'TSNR (Visual Cortex)'}`
- **Dictionary value**: `pandas.DataFrame`
  - Columns: `['Subject', 'Session', 'Pre-processing', 'NORDIC', <metric column>]`
  - `<metric column>` is either `TSNR (Full Brain)` or `TSNR (Visual Cortex)`
  - One row per `(Subject, Session, Pre-processing, NORDIC)` available in the source data.

### Downstream notebooks that load this file

- `N10a_TSNR_and_pBOLD_BarPlots_Figure04.evaluation.ipynb`
- `N10b_TSNR_and_pBOLD_BarPlots_SuppFig03.discovery.ipynb`
- `N11_Figure05_ScanLevel_ProblemIdentification.ipynb`


***
# 6. Gather Global Signal Timeseries

First, we load the estimates of Kappa and Rho for the GS

In [15]:
if DATASET == 'evaluation':
    kappa_rho_df = pd.read_csv(f'./summary_files/{DATASET}_gs_kappa_rho.csv', index_col=[0,1])
    print("++ INFO: The shape of kappa_rho_df is %s" % str(kappa_rho_df.shape))
else:
    kappa_rho_df = None

++ INFO: The shape of kappa_rho_df is (439, 3)


In [16]:
gs_sf_path = f'./summary_files/{DATASET}_GS_info_and_ts.pkl'
print('++ INFO: Global Signal will be saved to %s' % gs_sf_path)

++ INFO: Global Signal will be saved to ./summary_files/evaluation_GS_info_and_ts.pkl


In [17]:
fMRI_Sampling_Rate      = FMRI_TRS[DATASET]
fMRI_Preproc_Nsamples   = FMRI_FINAL_NUM_SAMPLES[DATASET]
fMRI_Ndiscard_samples   = NUM_DISCARDED_VOLUMES[DATASET]
fMRI_Preproc_Start_Time = str(float(fMRI_Sampling_Rate.replace('s','')) * int(fMRI_Ndiscard_samples))+'s'
print('++ fMRI Sampling Rate            = %s' % fMRI_Sampling_Rate)
print('++ fMRI Final Number of Samples  = %d Acquisitions' % fMRI_Preproc_Nsamples)
print('++ fMRI Number discarded Samples = %d Acquisitions' % fMRI_Ndiscard_samples)
print('++ fMRI Preproc Start Time       = %s' % fMRI_Preproc_Start_Time)

++ fMRI Sampling Rate            = 3s
++ fMRI Final Number of Samples  = 201 Acquisitions
++ fMRI Number discarded Samples = 3 Acquisitions
++ fMRI Preproc Start Time       = 9.0s


In [18]:
fMRI_Preproc_index = pd.timedelta_range(start=fMRI_Preproc_Start_Time, periods=fMRI_Preproc_Nsamples, freq=fMRI_Sampling_Rate)
fMRI_Preproc_index[0:3]

TimedeltaIndex(['0 days 00:00:09', '0 days 00:00:12', '0 days 00:00:15'], dtype='timedelta64[ns]', freq='3s')

In [19]:
gs_df_dict = {}
for sbj,ses in tqdm(ds_index):
    if DATASET == 'evaluation':
        # Load All the GS versions (each echo time and optimally combined)
        gs_e01_path = osp.join(PRCS_DATA_DIR,sbj,f'D03_Preproc_{ses}_NORDIC-off',f'pb03.{sbj}.r01.e01.volreg.GS.1D')
        gs_e02_path = osp.join(PRCS_DATA_DIR,sbj,f'D03_Preproc_{ses}_NORDIC-off',f'pb03.{sbj}.r01.e02.volreg.GS.1D')
        gs_e03_path = osp.join(PRCS_DATA_DIR,sbj,f'D03_Preproc_{ses}_NORDIC-off',f'pb03.{sbj}.r01.e03.volreg.GS.1D')
        gs_OC_path  = osp.join(PRCS_DATA_DIR,sbj,f'D03_Preproc_{ses}_NORDIC-off',f'pb06.{sbj}.r01.tedana_fastica_OC.GS.1D')
        gs_e01 = np.loadtxt(gs_e01_path)
        gs_e02 = np.loadtxt(gs_e02_path)
        gs_e03 = np.loadtxt(gs_e03_path)
        gs_OC  = np.loadtxt(gs_OC_path)
        gs_df = pd.DataFrame([gs_OC,gs_e01,gs_e02,gs_e03],index=['OC','TE1','TE2','TE3']).T
        gs_df = gs_df.infer_objects()
        gs_df.index = fMRI_Preproc_index
        gs_df.index.name = 'Time'
        gs_df.columns.name='Echo Time'
        # Transform to units of Signal Percent Change
        gs_df_spc = 100*(gs_df-gs_df.mean())/gs_df.mean()
        gs_df_dict[(sbj,ses,'gs_ts')] = gs_df_spc

        # Create Dataframe with Gs properties of interest
        GS_phys_match_file = osp.join(PRCS_DATA_DIR,sbj,f'D03_Preproc_{ses}_NORDIC-off',f'pb03.{sbj}.r01.e02.volreg.GS.PhysioModeling.pkl')
        try:
            with open(GS_phys_match_file, 'rb') as f:
                loaded_dict = pickle.load(f)
            gs_adjr2_physio = float(loaded_dict['model'].rsquared_adj)
        except:
            gs_adjr2_physio = None
        gs_kappa        = float(kappa_rho_df.loc[(sbj,ses),'kappa (GS)'])
        gs_rho          = float(kappa_rho_df.loc[(sbj,ses),'rho (GS)'])
        gs_df_metrics   = pd.DataFrame([gs_adjr2_physio,gs_kappa,gs_rho],index=['Adj R2 Physio','kappa','rho'],columns=['GS']).T
        gs_df_dict[sbj,ses,'gs_metrics'] = gs_df_metrics
    else:
        gs_df_dict[(sbj,ses,'gs_ts')] = None
        gs_df_dict[sbj,ses,'gs_metrics'] = None

  0%|          | 0/439 [00:00<?, ?it/s]

In [20]:
with open(gs_sf_path, 'wb') as f:
    pickle.dump(gs_df_dict, f)

---
# 4. Gather Tedana Outputs

## 4.1. TEDANA Statistics
This section summarizes TEDANA component counts and explained-variance metrics per scan/scenario.
Non-TEDANA pipelines are explicitly populated with `NaN` for these fields.


In [21]:
Tedana_QC_path = osp.join(CODE_DIR,'notebooks','summary_files',f'{DATASET}_Tedana_QC.pkl')
print('++ Tedana basic statistics will be saved in %s' % Tedana_QC_path)

++ Tedana basic statistics will be saved in /data/SFIMJGC_HCP7T/BCBL2024/me_staticfc/code/notebooks/summary_files/evaluation_Tedana_QC.pkl


Create empty data structures

In [22]:
%%time
Tedana_QC = {}
print(f"++ INFO: Loading all Tedana_QC for the {DATASET} into memory...")
for tedana_metric in ['#ICs (All)','#ICs (Likely BOLD)','#ICs (Unlikely BOLD)','Var. Exp. (Likely BOLD)','Var. Exp. (Unlikely BOLD)']:
    aux_df = pd.DataFrame(columns=['Subject','Session','Pre-processing','m-NORDIC',tedana_metric])
    aux_df.set_index(['Subject','Session','Pre-processing','m-NORDIC'], inplace=True)
    Tedana_QC['cov',tedana_metric] = aux_df

++ INFO: Loading all Tedana_QC for the evaluation into memory...
CPU times: user 10.1 ms, sys: 94 μs, total: 10.2 ms
Wall time: 9.64 ms


In [24]:
%%time
print(f"++ INFO: Loading all FC matrices for the {DATASET} into memory...")
for sbj,ses in tqdm(list(ds_index)):
    for nordic in ['off','on']:
        for pp in pp_opts.values():
            if 'Tedana' not in pp:
                Tedana_QC['cov','#ICs (All)'].loc[sbj,ses,pp,nordic]                = np.nan
                Tedana_QC['cov','#ICs (Likely BOLD)'].loc[sbj,ses,pp,nordic]        = np.nan
                Tedana_QC['cov','#ICs (Unlikely BOLD)'].loc[sbj,ses,pp,nordic]      = np.nan
                Tedana_QC['cov','Var. Exp. (Likely BOLD)'].loc[sbj,ses,pp,nordic]   = np.nan
                Tedana_QC['cov','Var. Exp. (Unlikely BOLD)'].loc[sbj,ses,pp,nordic] = np.nan
            else:
                tedana_type = pp.split('ALL_Tedana-')[1]
                d_folder    = f'D03_Preproc_{ses}_NORDIC-{nordic}'
                ica_metrics_path = osp.join(PRCS_DATA_DIR,sbj,d_folder,f'tedana_{tedana_type}','ica_metrics.tsv')
                ica_metrics              = pd.read_csv(ica_metrics_path, sep='\t').set_index('Component')
                likely_bold_components   = list(ica_metrics[ica_metrics['classification_tags']=='Likely BOLD'].index)
                unlikely_bold_components = list(ica_metrics[ica_metrics['classification_tags']=='Unlikely BOLD'].index)
                
                Tedana_QC['cov','#ICs (All)'].loc[sbj,ses,pp,nordic]              = ica_metrics.shape[0]
                Tedana_QC['cov','#ICs (Likely BOLD)'].loc[sbj,ses,pp,nordic]      = len(likely_bold_components)
                Tedana_QC['cov','#ICs (Unlikely BOLD)'].loc[sbj,ses,pp,nordic]    = len(unlikely_bold_components)
                Tedana_QC['cov','Var. Exp. (Likely BOLD)'].loc[sbj,ses,pp,nordic] = ica_metrics.loc[likely_bold_components,'variance explained'].sum().round(2)
                Tedana_QC['cov','Var. Exp. (Unlikely BOLD)'].loc[sbj,ses,pp,nordic] = ica_metrics.loc[unlikely_bold_components,'variance explained'].sum().round(2)
print('++ INFO: Saving to disk...',end='')
with open(Tedana_QC_path, 'wb') as f:
    pickle.dump(Tedana_QC, f)
print('[DONE]')

++ INFO: Loading all FC matrices for the evaluation into memory...


  0%|          | 0/439 [00:00<?, ?it/s]

++ INFO: Saving to disk...[DONE]
CPU times: user 47.4 s, sys: 908 ms, total: 48.4 s
Wall time: 51.9 s


### Output Schema: `{DATASET}_Tedana_QC.pkl`

- **Path**: `code/notebooks/summary_files/{DATASET}_Tedana_QC.pkl`
- **Serialized object type**: `dict`
- **Dictionary key (2-tuple)**:
  - `('cov', tedana_metric)`
  - `tedana_metric` in:
    - `'#ICs (All)'`
    - `'#ICs (Likely BOLD)'`
    - `'#ICs (Unlikely BOLD)'`
    - `'Var. Exp. (Likely BOLD)'`
    - `'Var. Exp. (Unlikely BOLD)'`
- **Dictionary value**: `pandas.DataFrame`
  - Columns: `['Subject', 'Session', 'Pre-processing', 'NORDIC', <tedana_metric>]`
  - For non-TEDANA pipelines, the metric field is populated with `NaN` by design.

### Downstream usage note

A repository-wide notebook scan currently found no other notebook (outside N08) directly loading `{DATASET}_Tedana_QC.pkl`.


## 4.2. ICA Timeseries and Additional Statistics

In [25]:
fMRI_Sampling_Rate      = FMRI_TRS[DATASET]
fMRI_Preproc_Nsamples   = FMRI_FINAL_NUM_SAMPLES[DATASET]
fMRI_Ndiscard_samples   = NUM_DISCARDED_VOLUMES[DATASET]
fMRI_Preproc_Start_Time = str(float(fMRI_Sampling_Rate.replace('s','')) * int(fMRI_Ndiscard_samples))+'s'
print('++ fMRI Sampling Rate            = %s' % fMRI_Sampling_Rate)
print('++ fMRI Final Number of Samples  = %d Acquisitions' % fMRI_Preproc_Nsamples)
print('++ fMRI Number discarded Samples = %d Acquisitions' % fMRI_Ndiscard_samples)
print('++ fMRI Preproc Start Time       = %s' % fMRI_Preproc_Start_Time)

++ fMRI Sampling Rate            = 3s
++ fMRI Final Number of Samples  = 201 Acquisitions
++ fMRI Number discarded Samples = 3 Acquisitions
++ fMRI Preproc Start Time       = 9.0s


In [26]:
fMRI_Preproc_index = pd.timedelta_range(start=fMRI_Preproc_Start_Time, periods=fMRI_Preproc_Nsamples, freq=fMRI_Sampling_Rate)
fMRI_Preproc_index[0:3]

TimedeltaIndex(['0 days 00:00:09', '0 days 00:00:12', '0 days 00:00:15'], dtype='timedelta64[ns]', freq='3s')

In [27]:
ica_sf_ts_path = f'./summary_files/{DATASET}_ICAs.pkl'
print('++INFO: IC Timeseries and additional stats saved to %s' % ica_sf_ts_path)

++INFO: IC Timeseries and additional stats saved to ./summary_files/evaluation_ICAs.pkl


In [28]:
ica_dict = {}
for sbj,ses in tqdm(ds_index):
    # Load IC Timeseries
    ic_ts_path         = osp.join(PRCS_DATA_DIR,sbj,f'D03_Preproc_{ses}_NORDIC-off','tedana_fastica',f'ica_mixing.tsv')
    ic_ts              = pd.read_csv(ic_ts_path, sep='\t')
    ic_ts              = ic_ts.infer_objects()
    ic_ts.index        = fMRI_Preproc_index
    ic_ts.index.name   = 'Time'
    ic_ts.columns.name = 'Components'
    ica_dict[(sbj,ses,'ic_ts')] = ic_ts
    # Load IC Properties
    ic_metrics = pd.read_csv(osp.join(PRCS_DATA_DIR,sbj,f'D03_Preproc_{ses}_NORDIC-off','tedana_fastica',f'ica_metrics.tsv'),sep='\t', index_col=0)
    ic_metrics.index.name='Name'
    ic_metrics               = ic_metrics.round(2)[['kappa','rho','variance explained','classification_tags']]
    if DATASET == 'evaluation':
        ic_metrics['corrwithGS'] = ic_ts.corrwith(gs_df_dict[sbj,ses,'gs_ts']['OC'])
        ic_metrics.columns = ['kappa','rho','varepx','label','R(ic,GS)']
    else:
        ic_metrics['corrwithGS'] = np.nan
        ic_metrics.columns = ['kappa','rho','varepx','label','R(ic,GS)']
    ica_dict[(sbj,ses,'ic_metrics')] = ic_metrics

  0%|          | 0/439 [00:00<?, ?it/s]

In [29]:
with open(ica_sf_ts_path, 'wb') as f:
    pickle.dump(ica_dict, f)

---
# 5. Physiological Traces

## 5.1. Gather Respiratory and Cardiac Recordings

This section builds a dictionary of per-scan cardiac and respiratory time series (evaluation dataset only, when source files exist).


In [30]:
physio_ts_sf_path = f'./summary_files/{DATASET}_Physiological_Timeseries.pkl'
print('++ INFO: Physiological Recordings will be saved to: %s' % physio_ts_sf_path)

++ INFO: Physiological Recordings will be saved to: ./summary_files/evaluation_Physiological_Timeseries.pkl


In [31]:
%%time

physio_dict = {}
for sbj,ses in tqdm(ds_index):
    if DATASET == 'evaluation':
        slibase_file = osp.join(PRCS_DATA_DIR,sbj,'D06_Physio',f'{sbj}_{ses}_task-rest_echo-1_slibase.1D')
        if osp.exists(slibase_file):
            # Load Physio by creating Afni RetroObj
            phys_file = osp.join(DOWNLOAD_DIR,sbj,ses,'func',f'{sbj}_{ses}_task-rest_physio.tsv.gz')
            json_file = osp.join(DOWNLOAD_DIR,sbj,ses,'func',f'{sbj}_{ses}_task-rest_physio.json')
            dset_epi  = osp.join(DOWNLOAD_DIR,sbj,ses,'func',f'{sbj}_{ses}_task-rest_echo-1_bold.nii.gz')
            input_line = ['./physio_calc.py', '-phys_file', phys_file, '-phys_json', json_file, '-dset_epi', dset_epi, 
                            '-prefilt_mode', 'median', '-prefilt_max_freq', '50', '-verb','0']
            args_orig  = copy.deepcopy(input_line)
            args_dict  = lpo.main_option_processing( input_line )
            retobj     = lpr.retro_obj( args_dict, args_orig=args_orig )
            physio_start_time = retobj.start_time
            # Extract Cardiac Timseries
            card_end_time  = retobj.data['card'].end_time
            card_nsamples  = retobj.data['card'].ts_orig.shape[0]
            card_samp_rate = retobj.data['card'].samp_rate
            card_index     = pd.timedelta_range(start=str(physio_start_time)+"s", periods=card_nsamples, freq=str(card_samp_rate)+"s")
            card_df        = pd.DataFrame(retobj.data['card'].ts_orig, index=card_index,columns=['PGG'])
            card_df.index.name = 'Time'

            # Extract Respiratory Timeseries
            resp_end_time  = retobj.data['resp'].end_time
            resp_nsamples  = retobj.data['resp'].ts_orig.shape[0]
            resp_samp_rate = retobj.data['resp'].samp_rate
            resp_index     = pd.timedelta_range(start=str(physio_start_time)+"s", periods=resp_nsamples, freq=str(resp_samp_rate)+"s")
            resp_df        = pd.DataFrame(retobj.data['resp'].ts_orig, index=resp_index,columns=['Respiration'])
            resp_df.index.name = 'Time'

            #Add to final dictionary
            physio_dict[(sbj,ses,'card')] = card_df
            physio_dict[(sbj,ses,'resp')] = resp_df
        else:
            physio_dict[(sbj,ses,'card')] = None
            physio_dict[(sbj,ses,'resp')] = None
    else:
        physio_dict[(sbj,ses,'card')] = None
        physio_dict[(sbj,ses,'resp')] = None

  0%|          | 0/439 [00:00<?, ?it/s]

++ Good: No items found in bad nan list
++ Good: No items found in bad null list
++ No downsampling needed: data sampling (40 Hz) < limit sampling freq (50.0 Hz)
++ No downsampling needed: data sampling (40 Hz) < limit sampling freq (50.0 Hz)
+* NB: card data prefilter window size is small (4 time points), because
       data samp freq = 40 Hz
       prefilt window = 0.1 s
   Are you sure that's what you want?
++ No downsampling needed: data sampling (40 Hz) < limit sampling freq (50.0 Hz)
++ No downsampling needed: data sampling (40 Hz) < limit sampling freq (50.0 Hz)
++ (card) Start time physio : 0.000000
++ (card) End times of physio and MRI:
   physio end time      : 612.150000
   final MRI slice time : 611.934784
++ (resp) Start time physio : 0.000000
++ (resp) End times of physio and MRI:
   physio end time      : 612.150000
   final MRI slice time : 611.934784
++ Good: No items found in bad nan list
++ Good: No items found in bad null list
++ No downsampling needed: data samplin

In [33]:
with open(physio_ts_sf_path, 'wb') as f:
    pickle.dump(physio_dict, f)

### Output Schema: `physio_dict` (intended for `{DATASET}_Physiological_Timeseries.pkl`)

- **Intended path variable**: `./summary_files/{DATASET}_Physiological_Timeseries.pkl`
- **In-memory object type**: `dict`
- **Dictionary key (3-tuple)**:
  - `(Subject, Session, signal_type)` where `signal_type` is `'card'` or `'resp'`
- **Dictionary value**:
  - For available evaluation scans:
    - `'card'`: `pandas.DataFrame` with one column `['PGG']` and `TimedeltaIndex` named `'Time'`
    - `'resp'`: `pandas.DataFrame` with one column `['Respiration']` and `TimedeltaIndex` named `'Time'`
  - Missing/unavailable scans (or discovery dataset): `None`

### Downstream notebooks that expect this file

- `N11_Figure05_ScanLevel_ProblemIdentification.ipynb` loads `./summary_files/{DATASET}_Physiological_Timeseries.pkl` with `pickle.load`.
- `NN05a_Results_Dashboard.ipynb` loads a different cache artifact (`./cache/{DATASET}_{CENSORING_MODE}_Physiological_Timeseries.pkl`), not the N08 summary-file path.


## Summary File Dependency Map

| N08 artifact | Produced in section | Loaded by notebooks |
|---|---|---|
| `{DATASET}_FC.pkl` | Section 1 | `N09`, `N11`, `N13` |
| `{DATASET}_pBOLD.nc` | Section 2 | `N09`, `N10a`, `N10b`, `N11` |
| `{DATASET}_TSNR.pkl` | Section 3 | `N10a`, `N10b`, `N11` |
| `{DATASET}_Tedana_QC.pkl` | Section 4 | No direct loaders found outside N08 (current scan) |
| `{DATASET}_Physiological_Timeseries.pkl` (intended) | Section 5 | `N11` expects this summary-file path |

This map is based on a code-level scan of notebook cells in `code/notebooks/*.ipynb`.


## 5.2. Gather Physiological Regressors


In [34]:
physio_regressors_sf_path = f'./summary_files/{DATASET}_Physiological_Regressors.pkl'
print('++ INFO: Physiological Regressors will be saved to %s' % physio_regressors_sf_path)


++ INFO: Physiological Regressors will be saved to ./summary_files/evaluation_Physiological_Regressors.pkl


In [35]:
physio_reg_dict = {}
for sbj,ses in tqdm(ds_index):
    if DATASET == 'evaluation':
        slibase_path = osp.join(PRCS_DATA_DIR,sbj,'D06_Physio',f'{sbj}_{ses}_task-rest_echo-1_slibase.1D')
        if osp.exists(slibase_path):
            slibase_obj  = Afni1D(slibase_path)
            slibase_df   = pd.read_csv(slibase_path, comment='#', delimiter=' +', header=None, engine='python')
            slibase_df.columns=slibase_obj.labels
            slibase_df=slibase_df[3::].reset_index(drop=True)
            slibase_df.index = fMRI_Preproc_index
            slibase_df.index.name = 'Time'
            slibase_df.columns.name = 'Regressors'
            # Load list of selected regressors for the GS
            GS_phys_match_file = osp.join(PRCS_DATA_DIR,sbj,f'D03_Preproc_{ses}_NORDIC-off',f'pb03.{sbj}.r01.e02.volreg.GS.PhysioModeling.pkl')
            with open(GS_phys_match_file, 'rb') as f:
                loaded_dict = pickle.load(f)
                selected_physio_regs = loaded_dict['selected_regs']
            selected_RVT_regs  = [r for r in selected_physio_regs if 'rvt' in r]
            selected_card_regs = [r for r in selected_physio_regs if 'card' in r]
            selected_resp_regs = [r for r in selected_physio_regs if 'resp' in r]
            physio_reg_dict[(sbj,ses,'RVT_regs')]  = slibase_df[selected_RVT_regs]
            physio_reg_dict[(sbj,ses,'card_regs')] = slibase_df[selected_card_regs]
            physio_reg_dict[(sbj,ses,'resp_regs')] = slibase_df[selected_resp_regs]
        else:
            physio_reg_dict[(sbj,ses,'RVT_regs')] = None
            physio_reg_dict[(sbj,ses,'card_regs')] = None
            physio_reg_dict[(sbj,ses,'resp_regs')] = None
    else:
        physio_reg_dict[(sbj,ses,'RVT_regs')] = None
        physio_reg_dict[(sbj,ses,'card_regs')] = None
        physio_reg_dict[(sbj,ses,'resp_regs')] = None

  0%|          | 0/439 [00:00<?, ?it/s]

In [36]:
with open(physio_regressors_sf_path, 'wb') as f:
    pickle.dump(physio_reg_dict, f)